# Research Experiment: Interpretable Deep RL for e-Otinish Complaint Routing

**Author**: Alseitov Olzhas

Run all cells. Final cell saves results_report.txt — copy and send to Claude.

## Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # For non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time
import os
import json
import warnings
warnings.filterwarnings('ignore')
from collections import defaultdict, deque
from scipy import stats

# Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score, precision_score, recall_score)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

plt.style.use('seaborn-v0_8-whitegrid')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

RESULTS = {}  # Global results collector
FIGURES_DIR = "figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

print("=" * 70)
print("EXPERIMENT: Interpretable Deep RL for e-Otinish Complaint Routing")
print("=" * 70)

## Data Generation (matching real e-Otinish distribution)

In [ ]:
TOPIC_DISTRIBUTION = {
    "legal_issues": 0.237, "government_services": 0.150,
    "corruption": 0.122, "safety": 0.086, "social_support": 0.072,
    "housing_utilities": 0.052, "transport": 0.039, "employment": 0.039,
    "judicial_system": 0.020, "economy": 0.020, "governance": 0.019,
    "finance": 0.019, "construction": 0.017, "taxes": 0.017,
    "land_relations": 0.017, "prosecutor": 0.016, "citizen_appeals": 0.015,
    "legislation": 0.015, "education": 0.014, "healthcare": 0.013
}

TOPIC_TO_DEPT = {
    "legal_issues": "legal", "judicial_system": "legal",
    "prosecutor": "legal", "legislation": "legal",
    "government_services": "admin_services", "citizen_appeals": "admin_services",
    "governance": "admin_services",
    "corruption": "anti_corruption", "finance": "anti_corruption",
    "safety": "public_safety", "transport": "public_safety",
    "social_support": "social_welfare", "healthcare": "social_welfare",
    "education": "social_welfare",
    "housing_utilities": "housing", "construction": "housing",
    "land_relations": "housing",
    "employment": "economic", "economy": "economic", "taxes": "economic"
}

DEPARTMENTS = ["legal", "admin_services", "anti_corruption",
               "public_safety", "social_welfare", "housing", "economic"]

# Extended text templates for richer TF-IDF
TEXT_TEMPLATES = {
    "legal_issues": [
        "Legal dispute regarding unfair court decision and delayed investigation process",
        "Rights violation complaint about improper legal proceedings in district court",
        "Request for legal assistance regarding property dispute and inheritance conflict",
        "Complaint about delayed court hearing and lack of legal representation",
        "Appeal against administrative fine imposed without proper legal basis",
    ],
    "government_services": [
        "Slow processing of passport renewal application at government office",
        "Complaint about rude behavior of civil servants at registration center",
        "Delayed issuance of birth certificate and bureaucratic obstacles",
        "Digital government portal malfunction preventing service access",
        "Long waiting times at public service center for document processing",
    ],
    "corruption": [
        "Report of bribery demand by local government official for permit approval",
        "Corruption in procurement process involving public funds misuse",
        "Nepotism complaint regarding hiring practices in state organization",
        "Extortion by traffic police officer demanding payment to avoid fine",
        "Fraudulent use of government budget in construction project",
    ],
    "safety": [
        "Dangerous road conditions and lack of traffic signals at intersection",
        "Complaint about inadequate street lighting causing safety concerns",
        "Report of illegal construction threatening neighborhood safety",
        "Stray animal attacks in residential area with no animal control response",
        "Fire safety violations in commercial building near residential zone",
    ],
    "social_support": [
        "Delayed social benefit payments for disabled family member",
        "Complaint about denial of child support allowance despite eligibility",
        "Request for social housing assistance for single parent family",
        "Pension calculation error resulting in reduced monthly payments",
        "Lack of social services for elderly citizens in rural area",
    ],
    "housing_utilities": [
        "Heating system failure in apartment building during winter season",
        "Water supply interruption lasting several days without notice",
        "Complaint about excessive utility bills and incorrect meter readings",
        "Building management company failing to perform maintenance duties",
        "Elevator breakdown in residential building not repaired for months",
    ],
    "transport": [
        "Poor public transport coverage in newly developed residential area",
        "Bus route cancellation affecting daily commuters without alternative",
        "Dangerous potholes on main road damaging vehicles regularly",
        "Traffic congestion due to poor road planning and construction delays",
        "Complaint about taxi service overcharging and unsafe driving",
    ],
    "employment": [
        "Illegal termination from workplace without proper compensation",
        "Employer refusing to pay salary for three consecutive months",
        "Workplace safety violations putting employees at health risk",
        "Discrimination in hiring process based on age and gender",
        "Forced overtime without additional payment in violation of labor code",
    ],
    "judicial_system": [
        "Judge bias complaint regarding unfair treatment during trial proceedings",
        "Delayed execution of court decision lasting over six months",
    ],
    "economy": [
        "Small business registration obstacles and excessive bureaucratic requirements",
        "Monopolistic pricing practices by utility companies harming consumers",
    ],
    "governance": [
        "Local government unresponsive to community infrastructure requests",
        "Akim office failing to address public petition signed by residents",
    ],
    "finance": [
        "Bank refusing to process loan application without valid justification",
        "Financial fraud complaint involving unauthorized transactions",
    ],
    "construction": [
        "Illegal construction activity without proper building permits",
        "Construction company failing to complete contracted residential project",
    ],
    "taxes": [
        "Incorrect tax assessment requiring immediate correction and refund",
        "Tax office refusing to process legitimate deduction claim",
    ],
    "land_relations": [
        "Land boundary dispute with neighbor and incorrect cadastral records",
        "Illegal land seizure by local authority for commercial development",
    ],
    "prosecutor": [
        "Request for prosecutorial investigation into official misconduct",
        "Complaint about prosecutor inaction regarding reported crime",
    ],
    "citizen_appeals": [
        "Appeal regarding unanswered government inquiry exceeding legal deadline",
        "Formal complaint about lack of response from government agency",
    ],
    "legislation": [
        "Request for legislative review of outdated regulatory framework",
        "Complaint about contradictory legal provisions causing confusion",
    ],
    "education": [
        "School facilities deterioration affecting student learning environment",
        "Teacher shortage in rural school district impacting education quality",
    ],
    "healthcare": [
        "Hospital refusing treatment without unofficial payment demand",
        "Long waiting list for essential medical procedure in public hospital",
    ],
}

RISK_FLAGS = ["corruption_indicator", "children_involved", "threat_to_life",
              "disability_related", "repeated_complaint"]


def generate_dataset(n_samples=3000):
    """Generate realistic e-Otinish complaint dataset"""
    data = []
    topics = list(TOPIC_DISTRIBUTION.keys())
    weights = np.array(list(TOPIC_DISTRIBUTION.values()))
    weights = weights / weights.sum()  # Normalize to sum exactly to 1.0
    
    for i in range(n_samples):
        topic = np.random.choice(topics, p=weights)
        templates = TEXT_TEMPLATES.get(topic, ["General complaint about government service"])
        text = np.random.choice(templates)
        # Add some random variation
        variations = [
            " Urgent attention required.",
            " This has been ongoing for months.",
            " Multiple complaints filed with no resolution.",
            " Citizens are suffering due to this issue.",
            " Immediate action is needed to resolve this matter.",
            "",
            "",
        ]
        text += np.random.choice(variations)
        
        department = TOPIC_TO_DEPT[topic]
        sentiment = np.clip(np.random.normal(-0.3, 0.4), -1, 1)
        
        # Risk flags
        flags = []
        if topic == "corruption":
            if np.random.random() < 0.7:
                flags.append("corruption_indicator")
        if np.random.random() < 0.08:
            flags.append("children_involved")
        if topic == "safety" and np.random.random() < 0.3:
            flags.append("threat_to_life")
        if np.random.random() < 0.05:
            flags.append("disability_related")
        if np.random.random() < 0.15:
            flags.append("repeated_complaint")
        
        urgency = 0.3
        if "corruption_indicator" in flags: urgency += 0.3
        if "children_involved" in flags: urgency += 0.3
        if "threat_to_life" in flags: urgency += 0.4
        if "disability_related" in flags: urgency += 0.1
        if "repeated_complaint" in flags: urgency += 0.1
        urgency = min(urgency + np.random.normal(0, 0.1), 1.0)
        urgency = max(urgency, 0.0)
        
        data.append({
            "id": i, "text": text, "topic": topic,
            "department": department, "sentiment": round(sentiment, 3),
            "risk_flags": flags, "urgency": round(urgency, 3),
            "n_risk_flags": len(flags)
        })
    
    return pd.DataFrame(data)


print("\n[1/8] Generating dataset...")
df = generate_dataset(3000)
print(f"  Dataset: {len(df)} complaints, {df['department'].nunique()} departments, {df['topic'].nunique()} topics")

# Data analysis
RESULTS['dataset'] = {
    'n_samples': len(df),
    'n_topics': df['topic'].nunique(),
    'n_departments': df['department'].nunique(),
    'topic_distribution': df['topic'].value_counts().to_dict(),
    'department_distribution': df['department'].value_counts().to_dict(),
    'avg_urgency': float(df['urgency'].mean()),
    'high_risk_pct': float((df['n_risk_flags'] > 0).mean() * 100),
    'avg_sentiment': float(df['sentiment'].mean()),
}

## Feature Engineering

In [ ]:
print("\n[2/8] Feature Engineering...")

vectorizer = TfidfVectorizer(max_features=100, ngram_range=(1, 2), min_df=2)
text_features = vectorizer.fit_transform(df['text']).toarray()

dept_encoder = LabelEncoder()
dept_labels = dept_encoder.fit_transform(df['department'])
n_departments = len(dept_encoder.classes_)

ACTIONS = DEPARTMENTS + ["escalate", "request_clarification", "reject"]
n_actions = len(ACTIONS)


def create_state_features(row, dept_workload=None):
    """State vector: TF-IDF (100) + risk flags (5) + metadata (3) + workload (7) = 115"""
    text_vec = vectorizer.transform([row['text']]).toarray()[0]
    risk_vec = [1 if flag in row['risk_flags'] else 0 for flag in RISK_FLAGS]
    meta_vec = [row['sentiment'], row['urgency'], row['n_risk_flags'] / 5.0]
    if dept_workload is None:
        wl_vec = [0.0] * n_departments
    else:
        total = max(sum(dept_workload.values()), 1)
        wl_vec = [dept_workload.get(d, 0) / total for d in DEPARTMENTS]
    return np.concatenate([text_vec, risk_vec, meta_vec, wl_vec]).astype(np.float32)


state_dim = 100 + 5 + 3 + n_departments  # 115
print(f"  State dimension: {state_dim}")
print(f"  Action space: {n_actions} ({n_departments} depts + 3 special)")

# Prepare supervised data
print("  Building feature matrices...")
X_all = np.array([create_state_features(row) for _, row in df.iterrows()])
y_all = dept_labels

X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all)
train_idx, test_idx = train_test_split(range(len(df)), test_size=0.2, random_state=SEED, stratify=y_all)

df_train = df.iloc[train_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

print(f"  Train: {len(X_train)}, Test: {len(X_test)}")

## Baseline Models (Random Forest + SVM)

In [ ]:
print("\n[3/8] Training Baseline Models...")

# --- Random Forest ---
t0 = time.time()
rf_model = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1, max_depth=20)
rf_model.fit(X_train, y_train)
rf_train_time = time.time() - t0

t0 = time.time()
rf_pred = rf_model.predict(X_test)
rf_inference_time = (time.time() - t0) / len(X_test) * 1000  # ms per sample

rf_acc = accuracy_score(y_test, rf_pred)
rf_f1 = f1_score(y_test, rf_pred, average='weighted')
rf_report = classification_report(y_test, rf_pred, target_names=dept_encoder.classes_, output_dict=True)
rf_cm = confusion_matrix(y_test, rf_pred)

print(f"  Random Forest: Acc={rf_acc:.4f}, F1={rf_f1:.4f}, Train={rf_train_time:.2f}s")

# --- SVM ---
t0 = time.time()
svm_model = SVC(kernel='rbf', C=10.0, gamma='scale', random_state=SEED, decision_function_shape='ovr')
svm_model.fit(X_train, y_train)
svm_train_time = time.time() - t0

t0 = time.time()
svm_pred = svm_model.predict(X_test)
svm_inference_time = (time.time() - t0) / len(X_test) * 1000

svm_acc = accuracy_score(y_test, svm_pred)
svm_f1 = f1_score(y_test, svm_pred, average='weighted')
svm_report = classification_report(y_test, svm_pred, target_names=dept_encoder.classes_, output_dict=True)

print(f"  SVM:           Acc={svm_acc:.4f}, F1={svm_f1:.4f}, Train={svm_train_time:.2f}s")

RESULTS['baseline_rf'] = {
    'accuracy': rf_acc, 'f1': rf_f1, 'train_time': rf_train_time,
    'inference_ms': rf_inference_time,
    'report': classification_report(y_test, rf_pred, target_names=dept_encoder.classes_),
    'confusion_matrix': rf_cm.tolist()
}
RESULTS['baseline_svm'] = {
    'accuracy': svm_acc, 'f1': svm_f1, 'train_time': svm_train_time,
    'inference_ms': svm_inference_time,
    'report': classification_report(y_test, svm_pred, target_names=dept_encoder.classes_),
}

## RL Environment

In [ ]:
print("\n[4/8] Setting up RL Environment...")


class ComplaintRoutingEnv:
    """Custom Gym-style environment for complaint routing"""
    
    def __init__(self, dataframe, episode_length=50):
        self.df = dataframe
        self.episode_length = episode_length
        self.n_actions = n_actions
        self.state_dim = state_dim
        self.reset()
    
    def reset(self):
        self.step_count = 0
        self.dept_workload = {d: 0 for d in DEPARTMENTS}
        self.total_reward = 0
        self.correct_routes = 0
        self.urgent_handled = 0
        self.urgent_total = 0
        self.episode_complaints = self.df.sample(self.episode_length).reset_index(drop=True)
        
        row = self.episode_complaints.iloc[0]
        state = create_state_features(row, self.dept_workload)
        self.current_row = row
        return state
    
    def step(self, action):
        row = self.current_row
        correct_dept = row['department']
        correct_idx = DEPARTMENTS.index(correct_dept)
        is_urgent = row['urgency'] > 0.6
        
        if is_urgent:
            self.urgent_total += 1
        
        reward = 0.0
        
        if action < n_departments:
            chosen_dept = DEPARTMENTS[action]
            if chosen_dept == correct_dept:
                reward += 10.0
                self.correct_routes += 1
                if is_urgent:
                    reward += 5.0
                    self.urgent_handled += 1
            else:
                reward -= 5.0
                if is_urgent:
                    reward -= 5.0  # Extra penalty for misrouting urgent
            
            # Workload balance penalty
            self.dept_workload[chosen_dept] += 1
            total_wl = max(sum(self.dept_workload.values()), 1)
            wl_values = [self.dept_workload[d] / total_wl for d in DEPARTMENTS]
            gini = self._gini(wl_values)
            reward -= gini * 2.0  # Penalize imbalance
            
            # Risk flag bonus
            if "corruption_indicator" in row['risk_flags'] and chosen_dept == "anti_corruption":
                reward += 3.0
            if "threat_to_life" in row['risk_flags'] and chosen_dept == "public_safety":
                reward += 3.0
            if "children_involved" in row['risk_flags'] and chosen_dept == "social_welfare":
                reward += 3.0
                
        elif action == n_departments:  # escalate
            if is_urgent:
                reward += 3.0
                self.urgent_handled += 1
            else:
                reward -= 2.0
        elif action == n_departments + 1:  # request clarification
            reward -= 1.0
        elif action == n_departments + 2:  # reject
            reward -= 8.0
        
        self.total_reward += reward
        self.step_count += 1
        done = self.step_count >= self.episode_length
        
        if not done:
            row = self.episode_complaints.iloc[self.step_count]
            self.current_row = row
            next_state = create_state_features(row, self.dept_workload)
        else:
            next_state = np.zeros(self.state_dim, dtype=np.float32)
        
        info = {
            'correct': action == correct_idx if action < n_departments else False,
            'accuracy': self.correct_routes / self.step_count,
            'workload_gini': self._gini(list(self.dept_workload.values())),
            'urgent_handled': self.urgent_handled,
            'urgent_total': self.urgent_total,
        }
        
        return next_state, reward, done, info
    
    @staticmethod
    def _gini(values):
        if sum(values) == 0:
            return 0.0
        vals = np.array(values, dtype=float)
        vals = vals / vals.sum()
        n = len(vals)
        vals_sorted = np.sort(vals)
        index = np.arange(1, n + 1)
        return float((2 * np.sum(index * vals_sorted) - (n + 1) * np.sum(vals_sorted)) / (n * np.sum(vals_sorted) + 1e-10))


env_train = ComplaintRoutingEnv(df_train, episode_length=50)
env_test = ComplaintRoutingEnv(df_test, episode_length=50)
print(f"  Environment ready. State dim={state_dim}, Actions={n_actions}")

# Verify with random policy
state = env_train.reset()
total_r = 0
while True:
    action = np.random.randint(n_actions)
    state, r, done, info = env_train.step(action)
    total_r += r
    if done:
        break
print(f"  Random policy test: reward={total_r:.1f}, accuracy={info['accuracy']:.3f}")

## DQN Agent

In [ ]:
print("\n[5/8] Training DQN Agent...")


class QNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc4 = nn.Linear(hidden_dim // 2, action_dim)
        self.attention = nn.Linear(state_dim, state_dim)
        self.dropout = nn.Dropout(0.1)
    
    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.relu(self.fc3(x))
        return self.fc4(x)
    
    def get_attention(self, state):
        return torch.softmax(self.attention(state), dim=-1)


class DQNAgent:
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99,
                 epsilon=1.0, epsilon_min=0.05, epsilon_decay=0.995,
                 buffer_size=10000, batch_size=64, hidden_dim=256):
        self.action_dim = action_dim
        self.gamma = gamma
        self.epsilon = epsilon
        self.epsilon_min = epsilon_min
        self.epsilon_decay = epsilon_decay
        self.batch_size = batch_size
        
        self.q_network = QNetwork(state_dim, action_dim, hidden_dim)
        self.target_network = QNetwork(state_dim, action_dim, hidden_dim)
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=lr)
        
        self.buffer = deque(maxlen=buffer_size)
    
    def select_action(self, state, explore=True):
        if explore and np.random.random() < self.epsilon:
            return np.random.randint(self.action_dim)
        with torch.no_grad():
            q_vals = self.q_network(torch.FloatTensor(state).unsqueeze(0))
            return q_vals.argmax(1).item()
    
    def store_transition(self, s, a, r, s_next, done):
        self.buffer.append((s, a, r, s_next, done))
    
    def train_step(self):
        if len(self.buffer) < self.batch_size:
            return 0.0
        
        batch = random.sample(self.buffer, self.batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        
        states = torch.FloatTensor(np.array(states))
        actions = torch.LongTensor(actions).unsqueeze(1)
        rewards = torch.FloatTensor(rewards)
        next_states = torch.FloatTensor(np.array(next_states))
        dones = torch.FloatTensor(dones)
        
        q_values = self.q_network(states).gather(1, actions).squeeze()
        with torch.no_grad():
            next_q = self.target_network(next_states).max(1)[0]
            targets = rewards + self.gamma * next_q * (1 - dones)
        
        loss = F.mse_loss(q_values, targets)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.q_network.parameters(), 1.0)
        self.optimizer.step()
        
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay
        
        return loss.item()
    
    def update_target(self):
        self.target_network.load_state_dict(self.q_network.state_dict())


def train_agent(agent, env, n_episodes, target_update_freq=10, agent_type='dqn'):
    """Universal training loop for DQN"""
    rewards_hist, acc_hist, loss_hist, eps_hist = [], [], [], []
    
    for ep in range(n_episodes):
        state = env.reset()
        ep_reward, ep_loss, steps = 0, 0, 0
        
        while True:
            action = agent.select_action(state)
            next_state, reward, done, info = env.step(action)
            agent.store_transition(state, action, reward, next_state, float(done))
            loss = agent.train_step()
            ep_reward += reward
            ep_loss += loss
            steps += 1
            state = next_state
            if done:
                break
        
        if (ep + 1) % target_update_freq == 0:
            agent.update_target()
        
        rewards_hist.append(ep_reward)
        acc_hist.append(info['accuracy'])
        loss_hist.append(ep_loss / steps if steps > 0 else 0)
        eps_hist.append(agent.epsilon)
        
        if (ep + 1) % 50 == 0:
            avg_r = np.mean(rewards_hist[-20:])
            avg_a = np.mean(acc_hist[-20:])
            print(f"    Episode {ep+1}/{n_episodes}: Avg Reward={avg_r:.1f}, Acc={avg_a:.3f}, Eps={agent.epsilon:.3f}")
    
    return {'rewards': rewards_hist, 'accuracies': acc_hist, 
            'losses': loss_hist, 'epsilons': eps_hist}


# Train DQN
dqn_agent = DQNAgent(state_dim, n_actions, lr=1e-3, gamma=0.99, hidden_dim=256)
t0 = time.time()
dqn_results = train_agent(dqn_agent, env_train, n_episodes=300, target_update_freq=10)
dqn_train_time = time.time() - t0
print(f"  DQN Training complete in {dqn_train_time:.1f}s")

## REINFORCE (Policy Gradient) Agent

In [ ]:
print("\n[6/8] Training REINFORCE Agent...")


class PolicyNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)
    
    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        return F.softmax(self.fc3(x), dim=-1)


class REINFORCEAgent:
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99, hidden_dim=256):
        self.gamma = gamma
        self.action_dim = action_dim
        self.policy = PolicyNetwork(state_dim, action_dim, hidden_dim)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)
        self.log_probs = []
        self.rewards = []
    
    def select_action(self, state, explore=True):
        probs = self.policy(torch.FloatTensor(state).unsqueeze(0))
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        self.log_probs.append(dist.log_prob(action))
        return action.item()
    
    def store_reward(self, reward):
        self.rewards.append(reward)
    
    def update(self):
        R = 0
        returns = []
        for r in reversed(self.rewards):
            R = r + self.gamma * R
            returns.insert(0, R)
        
        returns = torch.FloatTensor(returns)
        if len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        
        loss = 0
        for log_prob, R in zip(self.log_probs, returns):
            loss -= log_prob * R
        
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy.parameters(), 1.0)
        self.optimizer.step()
        
        self.log_probs = []
        self.rewards = []
        return loss.item()


def train_reinforce(agent, env, n_episodes):
    rewards_hist, acc_hist, loss_hist = [], [], []
    
    for ep in range(n_episodes):
        state = env.reset()
        ep_reward = 0
        
        while True:
            action = agent.select_action(state)
            next_state, reward, done, info = env.step(action)
            agent.store_reward(reward)
            ep_reward += reward
            state = next_state
            if done:
                break
        
        loss = agent.update()
        rewards_hist.append(ep_reward)
        acc_hist.append(info['accuracy'])
        loss_hist.append(loss)
        
        if (ep + 1) % 50 == 0:
            avg_r = np.mean(rewards_hist[-20:])
            avg_a = np.mean(acc_hist[-20:])
            print(f"    Episode {ep+1}/{n_episodes}: Avg Reward={avg_r:.1f}, Acc={avg_a:.3f}")
    
    return {'rewards': rewards_hist, 'accuracies': acc_hist, 'losses': loss_hist}


reinforce_agent = REINFORCEAgent(state_dim, n_actions, lr=5e-4, gamma=0.99, hidden_dim=256)
t0 = time.time()
reinforce_results = train_reinforce(reinforce_agent, env_train, n_episodes=300)
reinforce_train_time = time.time() - t0
print(f"  REINFORCE Training complete in {reinforce_train_time:.1f}s")

## A2C (Advantage Actor-Critic) Agent

In [ ]:
print("\n[7/8] Training A2C Agent...")


class ActorCriticNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        self.actor = nn.Linear(hidden_dim, action_dim)
        self.critic = nn.Linear(hidden_dim, 1)
    
    def forward(self, state):
        shared = self.shared(state)
        policy = F.softmax(self.actor(shared), dim=-1)
        value = self.critic(shared)
        return policy, value


class A2CAgent:
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99,
                 entropy_coef=0.01, hidden_dim=256):
        self.gamma = gamma
        self.action_dim = action_dim
        self.entropy_coef = entropy_coef
        self.network = ActorCriticNetwork(state_dim, action_dim, hidden_dim)
        self.optimizer = optim.Adam(self.network.parameters(), lr=lr)
        self.transitions = []
    
    def select_action(self, state, explore=True):
        probs, value = self.network(torch.FloatTensor(state).unsqueeze(0))
        dist = torch.distributions.Categorical(probs)
        action = dist.sample()
        self.transitions.append({
            'log_prob': dist.log_prob(action),
            'value': value.squeeze(),
            'entropy': dist.entropy()
        })
        return action.item()
    
    def store_reward(self, reward):
        self.transitions[-1]['reward'] = reward
    
    def update(self):
        R = 0
        returns = []
        for t in reversed(self.transitions):
            R = t['reward'] + self.gamma * R
            returns.insert(0, R)
        
        returns = torch.FloatTensor(returns)
        if len(returns) > 1:
            returns = (returns - returns.mean()) / (returns.std() + 1e-8)
        
        policy_loss = 0
        value_loss = 0
        entropy_bonus = 0
        
        for t, R in zip(self.transitions, returns):
            advantage = R - t['value'].detach()
            policy_loss -= t['log_prob'] * advantage
            value_loss += F.mse_loss(t['value'], R)
            entropy_bonus -= t['entropy']
        
        loss = policy_loss + 0.5 * value_loss + self.entropy_coef * entropy_bonus
        
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.network.parameters(), 1.0)
        self.optimizer.step()
        
        self.transitions = []
        return loss.item()


def train_a2c(agent, env, n_episodes):
    rewards_hist, acc_hist, loss_hist = [], [], []
    
    for ep in range(n_episodes):
        state = env.reset()
        ep_reward = 0
        
        while True:
            action = agent.select_action(state)
            next_state, reward, done, info = env.step(action)
            agent.store_reward(reward)
            ep_reward += reward
            state = next_state
            if done:
                break
        
        loss = agent.update()
        rewards_hist.append(ep_reward)
        acc_hist.append(info['accuracy'])
        loss_hist.append(loss)
        
        if (ep + 1) % 50 == 0:
            avg_r = np.mean(rewards_hist[-20:])
            avg_a = np.mean(acc_hist[-20:])
            print(f"    Episode {ep+1}/{n_episodes}: Avg Reward={avg_r:.1f}, Acc={avg_a:.3f}")
    
    return {'rewards': rewards_hist, 'accuracies': acc_hist, 'losses': loss_hist}


a2c_agent = A2CAgent(state_dim, n_actions, lr=5e-4, gamma=0.99, hidden_dim=256)
t0 = time.time()
a2c_results = train_a2c(a2c_agent, env_train, n_episodes=300)
a2c_train_time = time.time() - t0
print(f"  A2C Training complete in {a2c_train_time:.1f}s")

## Comprehensive Evaluation

In [ ]:
print("\n[8/8] Comprehensive Evaluation...")


def evaluate_rl_agent(agent, env, n_episodes=30, agent_type='dqn'):
    """Evaluate RL agent on test environment"""
    all_rewards, all_accs, all_ginis = [], [], []
    all_urgent_rates = []
    dept_counts = defaultdict(int)
    correct_per_dept = defaultdict(int)
    total_per_dept = defaultdict(int)
    inference_times = []
    
    for _ in range(n_episodes):
        state = env.reset()
        ep_reward = 0
        
        while True:
            t0 = time.time()
            if agent_type == 'dqn':
                action = agent.select_action(state, explore=False)
            else:
                action = agent.select_action(state, explore=False)
            inference_times.append(time.time() - t0)
            
            next_state, reward, done, info = env.step(action)
            
            if action < n_departments:
                dept_counts[DEPARTMENTS[action]] += 1
                correct_dept = env.current_row['department'] if not done else None
            
            ep_reward += reward
            state = next_state
            if done:
                break
        
        all_rewards.append(ep_reward)
        all_accs.append(info['accuracy'])
        all_ginis.append(info['workload_gini'])
        if info['urgent_total'] > 0:
            all_urgent_rates.append(info['urgent_handled'] / info['urgent_total'])
    
    return {
        'mean_reward': np.mean(all_rewards),
        'std_reward': np.std(all_rewards),
        'mean_accuracy': np.mean(all_accs),
        'std_accuracy': np.std(all_accs),
        'mean_gini': np.mean(all_ginis),
        'mean_urgent_rate': np.mean(all_urgent_rates) if all_urgent_rates else 0.0,
        'inference_ms': np.mean(inference_times) * 1000,
        'dept_distribution': dict(dept_counts),
        'all_rewards': all_rewards,
        'all_accs': all_accs,
    }


# Evaluate all RL agents
dqn_eval = evaluate_rl_agent(dqn_agent, env_test, n_episodes=30, agent_type='dqn')
reinforce_eval = evaluate_rl_agent(reinforce_agent, env_test, n_episodes=30, agent_type='reinforce')
a2c_eval = evaluate_rl_agent(a2c_agent, env_test, n_episodes=30, agent_type='a2c')

# Random policy baseline
class RandomAgent:
    def __init__(self, n_actions):
        self.action_dim = n_actions
    def select_action(self, state, explore=True):
        return np.random.randint(self.action_dim)

random_agent = RandomAgent(n_actions)
random_eval = evaluate_rl_agent(random_agent, env_test, n_episodes=30, agent_type='random')

# Rule-based baseline
class RuleBasedAgent:
    """Simple rule-based: map highest TF-IDF features to departments"""
    def __init__(self):
        self.action_dim = n_actions
    def select_action(self, state, explore=True):
        # Use risk flags and urgency to decide
        corruption_flag = state[100]
        children_flag = state[101]
        threat_flag = state[102]
        urgency = state[106]
        
        if corruption_flag > 0.5:
            return DEPARTMENTS.index("anti_corruption")
        if threat_flag > 0.5:
            return DEPARTMENTS.index("public_safety")
        if children_flag > 0.5:
            return DEPARTMENTS.index("social_welfare")
        # Default: pick department with lowest workload
        wl = state[108:108+n_departments]
        return int(np.argmin(wl))

rule_agent = RuleBasedAgent()
rule_eval = evaluate_rl_agent(rule_agent, env_test, n_episodes=30, agent_type='rule')

print(f"  DQN:       Acc={dqn_eval['mean_accuracy']:.4f}, Reward={dqn_eval['mean_reward']:.1f}")
print(f"  REINFORCE: Acc={reinforce_eval['mean_accuracy']:.4f}, Reward={reinforce_eval['mean_reward']:.1f}")
print(f"  A2C:       Acc={a2c_eval['mean_accuracy']:.4f}, Reward={a2c_eval['mean_reward']:.1f}")
print(f"  Rule-based: Acc={rule_eval['mean_accuracy']:.4f}, Reward={rule_eval['mean_reward']:.1f}")
print(f"  Random:    Acc={random_eval['mean_accuracy']:.4f}, Reward={random_eval['mean_reward']:.1f}")

# Statistical significance (t-tests)
ttest_dqn_rf = stats.ttest_ind(dqn_eval['all_accs'], [rf_acc] * 30)
ttest_dqn_reinforce = stats.ttest_ind(dqn_eval['all_rewards'], reinforce_eval['all_rewards'])
ttest_dqn_a2c = stats.ttest_ind(dqn_eval['all_rewards'], a2c_eval['all_rewards'])

## Attention / Interpretability Analysis

In [ ]:
print("\nRunning interpretability analysis...")

feature_names = (list(vectorizer.get_feature_names_out()) + RISK_FLAGS + 
                 ['sentiment', 'urgency', 'n_risk_flags_norm'] +
                 [f'wl_{d}' for d in DEPARTMENTS])

interpretability_cases = []
for idx in range(min(10, len(df_test))):
    row = df_test.iloc[idx]
    state = create_state_features(row)
    state_tensor = torch.FloatTensor(state).unsqueeze(0)
    
    with torch.no_grad():
        attention_weights = dqn_agent.q_network.get_attention(state_tensor).squeeze().numpy()
        q_values = dqn_agent.q_network(state_tensor).squeeze().numpy()
    
    action = int(np.argmax(q_values))
    chosen = ACTIONS[action]
    correct = row['department']
    
    # Top features
    top_indices = np.argsort(attention_weights)[-10:][::-1]
    top_features = [(feature_names[i], float(attention_weights[i])) for i in top_indices]
    
    # Feature group importances
    text_importance = float(attention_weights[:100].mean())
    risk_importance = float(attention_weights[100:105].mean())
    meta_importance = float(attention_weights[105:108].mean())
    wl_importance = float(attention_weights[108:].mean())
    
    interpretability_cases.append({
        'complaint_text': row['text'][:100],
        'topic': row['topic'],
        'correct_dept': correct,
        'predicted': chosen,
        'is_correct': chosen == correct,
        'urgency': float(row['urgency']),
        'risk_flags': row['risk_flags'],
        'top_features': top_features,
        'group_importance': {
            'text': text_importance,
            'risk_flags': risk_importance,
            'metadata': meta_importance,
            'workload': wl_importance
        },
        'q_values': {ACTIONS[i]: float(q_values[i]) for i in range(len(ACTIONS))}
    })

## Hyperparameter Sensitivity Study

In [ ]:
print("\nRunning hyperparameter sensitivity...")

hp_results = {}
for lr in [1e-4, 1e-3, 1e-2]:
    agent_hp = DQNAgent(state_dim, n_actions, lr=lr, gamma=0.99, hidden_dim=256)
    res = train_agent(agent_hp, env_train, n_episodes=100, target_update_freq=10)
    eval_hp = evaluate_rl_agent(agent_hp, env_test, n_episodes=10, agent_type='dqn')
    hp_results[f'lr={lr}'] = {
        'final_reward': np.mean(res['rewards'][-20:]),
        'final_accuracy': np.mean(res['accuracies'][-20:]),
        'eval_accuracy': eval_hp['mean_accuracy'],
        'training_curve': res['rewards']
    }
    print(f"    lr={lr}: Final Acc={eval_hp['mean_accuracy']:.4f}")

for hidden in [128, 256, 512]:
    agent_hp = DQNAgent(state_dim, n_actions, lr=1e-3, gamma=0.99, hidden_dim=hidden)
    res = train_agent(agent_hp, env_train, n_episodes=100, target_update_freq=10)
    eval_hp = evaluate_rl_agent(agent_hp, env_test, n_episodes=10, agent_type='dqn')
    hp_results[f'hidden={hidden}'] = {
        'final_reward': np.mean(res['rewards'][-20:]),
        'final_accuracy': np.mean(res['accuracies'][-20:]),
        'eval_accuracy': eval_hp['mean_accuracy'],
        'training_curve': res['rewards']
    }
    print(f"    hidden={hidden}: Final Acc={eval_hp['mean_accuracy']:.4f}")

## Reward Ablation Study

In [ ]:
print("\nRunning reward ablation study...")

class AblationEnv(ComplaintRoutingEnv):
    def __init__(self, dataframe, reward_mode='full', episode_length=50):
        self.reward_mode = reward_mode
        super().__init__(dataframe, episode_length)
    
    def step(self, action):
        row = self.current_row
        correct_dept = row['department']
        correct_idx = DEPARTMENTS.index(correct_dept)
        is_urgent = row['urgency'] > 0.6
        
        if is_urgent:
            self.urgent_total += 1
        
        reward = 0.0
        
        if action < n_departments:
            chosen_dept = DEPARTMENTS[action]
            # Mode 1: Accuracy only
            if chosen_dept == correct_dept:
                reward += 10.0
                self.correct_routes += 1
                if is_urgent: self.urgent_handled += 1
            else:
                reward -= 5.0
            
            if self.reward_mode in ['accuracy_urgency', 'full']:
                if is_urgent:
                    reward += 5.0 if chosen_dept == correct_dept else -5.0
            
            if self.reward_mode == 'full':
                self.dept_workload[chosen_dept] += 1
                total_wl = max(sum(self.dept_workload.values()), 1)
                wl_vals = [self.dept_workload[d] / total_wl for d in DEPARTMENTS]
                reward -= self._gini(wl_vals) * 2.0
                
                if "corruption_indicator" in row['risk_flags'] and chosen_dept == "anti_corruption":
                    reward += 3.0
        else:
            if action < n_departments:
                self.dept_workload.setdefault(DEPARTMENTS[action], 0)
            if action == n_departments and is_urgent:
                reward += 3.0
                self.urgent_handled += 1
            else:
                reward -= 2.0
        
        self.total_reward += reward
        self.step_count += 1
        done = self.step_count >= self.episode_length
        
        if not done:
            row = self.episode_complaints.iloc[self.step_count]
            self.current_row = row
            next_state = create_state_features(row, self.dept_workload)
        else:
            next_state = np.zeros(self.state_dim, dtype=np.float32)
        
        info = {
            'correct': action == correct_idx if action < n_departments else False,
            'accuracy': self.correct_routes / self.step_count,
            'workload_gini': self._gini(list(self.dept_workload.values())),
            'urgent_handled': self.urgent_handled,
            'urgent_total': self.urgent_total,
        }
        return next_state, reward, done, info

ablation_results = {}
for mode in ['accuracy_only', 'accuracy_urgency', 'full']:
    env_abl = AblationEnv(df_train, reward_mode=mode)
    agent_abl = DQNAgent(state_dim, n_actions, lr=1e-3, gamma=0.99, hidden_dim=256)
    res = train_agent(agent_abl, env_abl, n_episodes=150, target_update_freq=10)
    eval_abl = evaluate_rl_agent(agent_abl, env_test, n_episodes=20, agent_type='dqn')
    ablation_results[mode] = {
        'accuracy': eval_abl['mean_accuracy'],
        'reward': eval_abl['mean_reward'],
        'gini': eval_abl['mean_gini'],
        'urgent_rate': eval_abl['mean_urgent_rate'],
        'training_rewards': res['rewards']
    }
    print(f"    {mode}: Acc={eval_abl['mean_accuracy']:.4f}, Gini={eval_abl['mean_gini']:.4f}")

## Generate ALL Figures

In [ ]:
print("\nGenerating figures...")

# Figure 1: Training Reward Curves (all RL methods)
fig, ax = plt.subplots(figsize=(10, 6))
window = 20
for label, data, color in [
    ('DQN', dqn_results['rewards'], '#2196F3'),
    ('REINFORCE', reinforce_results['rewards'], '#FF9800'),
    ('A2C', a2c_results['rewards'], '#4CAF50')
]:
    ax.plot(data, alpha=0.15, color=color)
    ax.plot(pd.Series(data).rolling(window).mean(), label=label, linewidth=2, color=color)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Total Episode Reward', fontsize=12)
ax.set_title('Training Convergence: Episode Reward', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig1_training_rewards.png', dpi=150)
plt.close()

# Figure 2: Training Accuracy Curves
fig, ax = plt.subplots(figsize=(10, 6))
for label, data, color in [
    ('DQN', dqn_results['accuracies'], '#2196F3'),
    ('REINFORCE', reinforce_results['accuracies'], '#FF9800'),
    ('A2C', a2c_results['accuracies'], '#4CAF50')
]:
    ax.plot(pd.Series(data).rolling(window).mean(), label=label, linewidth=2, color=color)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Routing Accuracy', fontsize=12)
ax.set_title('Training Convergence: Routing Accuracy', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig2_training_accuracy.png', dpi=150)
plt.close()

# Figure 3: Confusion Matrix (DQN)
fig, ax = plt.subplots(figsize=(8, 6))
# Build confusion matrix from DQN predictions on test set
dqn_preds = []
dqn_trues = []
for idx in range(min(300, len(df_test))):
    row = df_test.iloc[idx]
    state = create_state_features(row)
    action = dqn_agent.select_action(state, explore=False)
    if action < n_departments:
        dqn_preds.append(DEPARTMENTS[action])
        dqn_trues.append(row['department'])
if dqn_preds:
    cm_labels = sorted(set(dqn_trues + dqn_preds))
    cm = confusion_matrix(dqn_trues, dqn_preds, labels=cm_labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=cm_labels, yticklabels=cm_labels, ax=ax)
    ax.set_xlabel('Predicted Department', fontsize=11)
    ax.set_ylabel('True Department', fontsize=11)
    ax.set_title('DQN Agent Confusion Matrix', fontsize=13)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig3_confusion_matrix.png', dpi=150)
plt.close()

# Figure 4: Method Comparison Bar Chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
methods = ['Random', 'Rule-based', 'RF', 'SVM', 'DQN', 'REINFORCE', 'A2C']
accuracies = [
    random_eval['mean_accuracy'], rule_eval['mean_accuracy'],
    rf_acc, svm_acc,
    dqn_eval['mean_accuracy'], reinforce_eval['mean_accuracy'], a2c_eval['mean_accuracy']
]
rewards = [
    random_eval['mean_reward'], rule_eval['mean_reward'],
    0, 0,  # Supervised methods don't have episode reward
    dqn_eval['mean_reward'], reinforce_eval['mean_reward'], a2c_eval['mean_reward']
]
colors = ['#9E9E9E', '#795548', '#E91E63', '#9C27B0', '#2196F3', '#FF9800', '#4CAF50']

axes[0].bar(methods, accuracies, color=colors)
axes[0].set_ylabel('Routing Accuracy')
axes[0].set_title('Accuracy Comparison')
axes[0].tick_params(axis='x', rotation=45)

rl_methods = ['Random', 'Rule-based', 'DQN', 'REINFORCE', 'A2C']
rl_rewards = [random_eval['mean_reward'], rule_eval['mean_reward'],
              dqn_eval['mean_reward'], reinforce_eval['mean_reward'], a2c_eval['mean_reward']]
axes[1].bar(rl_methods, rl_rewards, color=['#9E9E9E', '#795548', '#2196F3', '#FF9800', '#4CAF50'])
axes[1].set_ylabel('Average Episode Reward')
axes[1].set_title('Reward Comparison (RL Methods)')
axes[1].tick_params(axis='x', rotation=45)

ginis = [random_eval['mean_gini'], rule_eval['mean_gini'],
         dqn_eval['mean_gini'], reinforce_eval['mean_gini'], a2c_eval['mean_gini']]
axes[2].bar(rl_methods, ginis, color=['#9E9E9E', '#795548', '#2196F3', '#FF9800', '#4CAF50'])
axes[2].set_ylabel('Workload Gini Coefficient')
axes[2].set_title('Fairness Comparison (lower=better)')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig4_comparison_bars.png', dpi=150)
plt.close()

# Figure 5: Workload Distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, eval_data, title in [
    (axes[0], dqn_eval, 'DQN'), (axes[1], reinforce_eval, 'REINFORCE'), (axes[2], a2c_eval, 'A2C')
]:
    dept_dist = eval_data.get('dept_distribution', {})
    if dept_dist:
        depts = list(dept_dist.keys())
        counts = list(dept_dist.values())
        ax.bar(depts, counts, color='steelblue')
    ax.set_title(f'{title} Workload Distribution')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig5_workload_distribution.png', dpi=150)
plt.close()

# Figure 6: Attention Heatmap (feature group importance)
fig, ax = plt.subplots(figsize=(10, 6))
if interpretability_cases:
    cases_matrix = []
    case_labels = []
    for i, case in enumerate(interpretability_cases[:8]):
        gi = case['group_importance']
        cases_matrix.append([gi['text'], gi['risk_flags'], gi['metadata'], gi['workload']])
        status = "✓" if case['is_correct'] else "✗"
        case_labels.append(f"Case {i+1} ({status})")
    
    sns.heatmap(np.array(cases_matrix), annot=True, fmt='.3f',
                xticklabels=['Text', 'Risk Flags', 'Metadata', 'Workload'],
                yticklabels=case_labels, cmap='YlOrRd', ax=ax)
    ax.set_title('DQN Attention: Feature Group Importance per Complaint', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig6_attention_heatmap.png', dpi=150)
plt.close()

# Figure 7: Reward Ablation Study
fig, ax = plt.subplots(figsize=(10, 6))
for mode, color in [('accuracy_only', '#E91E63'), ('accuracy_urgency', '#FF9800'), ('full', '#4CAF50')]:
    data = ablation_results[mode]['training_rewards']
    ax.plot(pd.Series(data).rolling(15).mean(), label=mode.replace('_', ' + '), linewidth=2, color=color)
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Episode Reward', fontsize=12)
ax.set_title('Reward Function Ablation Study', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig7_reward_ablation.png', dpi=150)
plt.close()

# Figure 8: Hyperparameter Sensitivity
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for lr_key in ['lr=0.0001', 'lr=0.001', 'lr=0.01']:
    if lr_key in hp_results:
        data = hp_results[lr_key]['training_curve']
        axes[0].plot(pd.Series(data).rolling(10).mean(), label=lr_key, linewidth=2)
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Episode Reward')
axes[0].set_title('Learning Rate Sensitivity')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

for h_key in ['hidden=128', 'hidden=256', 'hidden=512']:
    if h_key in hp_results:
        data = hp_results[h_key]['training_curve']
        axes[1].plot(pd.Series(data).rolling(10).mean(), label=h_key, linewidth=2)
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Episode Reward')
axes[1].set_title('Hidden Dimension Sensitivity')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig8_hyperparameter_sensitivity.png', dpi=150)
plt.close()

# Figure 9: Epsilon Decay
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(dqn_results['epsilons'], linewidth=2, color='#2196F3')
ax.set_xlabel('Episode', fontsize=12)
ax.set_ylabel('Epsilon', fontsize=12)
ax.set_title('DQN Epsilon Decay Schedule', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig9_epsilon_decay.png', dpi=150)
plt.close()

# Figure 10: Dataset Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
topic_counts = df['topic'].value_counts().head(10)
topic_counts.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Top 10 Complaint Topics')
axes[0].set_xlabel('Count')

dept_counts_plot = df['department'].value_counts()
dept_counts_plot.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Department Distribution')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/fig10_data_distribution.png', dpi=150)
plt.close()

print(f"  Saved 10 figures to {FIGURES_DIR}/")

## COMPILE FULL RESULTS REPORT

In [ ]:
print("\nCompiling results report...")

report = []
report.append("=" * 80)
report.append("FULL RESULTS REPORT: Interpretable Deep RL for e-Otinish Complaint Routing")
report.append("Author: Alseitov Olzhas")
report.append("=" * 80)

report.append("\n\n=== SECTION 1: DATASET STATISTICS ===")
report.append(f"Total complaints: {RESULTS['dataset']['n_samples']}")
report.append(f"Number of topics: {RESULTS['dataset']['n_topics']}")
report.append(f"Number of departments: {RESULTS['dataset']['n_departments']}")
report.append(f"Average urgency score: {RESULTS['dataset']['avg_urgency']:.3f}")
report.append(f"High-risk complaints (any flag): {RESULTS['dataset']['high_risk_pct']:.1f}%")
report.append(f"Average sentiment: {RESULTS['dataset']['avg_sentiment']:.3f}")
report.append(f"\nTopic distribution:")
for topic, count in sorted(RESULTS['dataset']['topic_distribution'].items(), key=lambda x: -x[1]):
    report.append(f"  {topic}: {count} ({count/RESULTS['dataset']['n_samples']*100:.1f}%)")
report.append(f"\nDepartment distribution:")
for dept, count in sorted(RESULTS['dataset']['department_distribution'].items(), key=lambda x: -x[1]):
    report.append(f"  {dept}: {count} ({count/RESULTS['dataset']['n_samples']*100:.1f}%)")

report.append("\n\n=== SECTION 2: BASELINE MODEL RESULTS ===")
report.append(f"\n--- Random Forest ---")
report.append(f"Accuracy: {rf_acc:.4f}")
report.append(f"F1 Score (weighted): {rf_f1:.4f}")
report.append(f"Training time: {rf_train_time:.2f}s")
report.append(f"Inference time: {rf_inference_time:.4f} ms/sample")
report.append(f"\nClassification Report:\n{RESULTS['baseline_rf']['report']}")
report.append(f"\nConfusion Matrix:\n{np.array(RESULTS['baseline_rf']['confusion_matrix'])}")

report.append(f"\n--- SVM ---")
report.append(f"Accuracy: {svm_acc:.4f}")
report.append(f"F1 Score (weighted): {svm_f1:.4f}")
report.append(f"Training time: {svm_train_time:.2f}s")
report.append(f"Inference time: {svm_inference_time:.4f} ms/sample")
report.append(f"\nClassification Report:\n{RESULTS['baseline_svm']['report']}")

report.append("\n\n=== SECTION 3: RL TRAINING RESULTS ===")
for name, results, train_time in [
    ('DQN', dqn_results, dqn_train_time),
    ('REINFORCE', reinforce_results, reinforce_train_time),
    ('A2C', a2c_results, a2c_train_time)
]:
    report.append(f"\n--- {name} ---")
    report.append(f"Episodes trained: {len(results['rewards'])}")
    report.append(f"Training time: {train_time:.1f}s")
    report.append(f"Final avg reward (last 20 eps): {np.mean(results['rewards'][-20:]):.2f}")
    report.append(f"Final avg accuracy (last 20 eps): {np.mean(results['accuracies'][-20:]):.4f}")
    report.append(f"Best episode reward: {max(results['rewards']):.2f}")
    report.append(f"Reward trajectory (every 50 eps):")
    for i in range(0, len(results['rewards']), 50):
        chunk = results['rewards'][i:i+50]
        report.append(f"  Episodes {i}-{i+50}: mean={np.mean(chunk):.2f}, std={np.std(chunk):.2f}")

report.append("\n\n=== SECTION 4: EVALUATION RESULTS (Test Set, 30 episodes) ===")
report.append(f"\n{'Method':<15} {'Accuracy':>10} {'Reward':>10} {'Gini':>8} {'Urgent%':>10} {'Infer(ms)':>10}")
report.append("-" * 65)
for name, ev in [
    ('Random', random_eval), ('Rule-based', rule_eval),
    ('RF', {'mean_accuracy': rf_acc, 'mean_reward': 0, 'mean_gini': 0, 'mean_urgent_rate': 0, 'inference_ms': rf_inference_time}),
    ('SVM', {'mean_accuracy': svm_acc, 'mean_reward': 0, 'mean_gini': 0, 'mean_urgent_rate': 0, 'inference_ms': svm_inference_time}),
    ('DQN', dqn_eval), ('REINFORCE', reinforce_eval), ('A2C', a2c_eval)
]:
    report.append(f"{name:<15} {ev['mean_accuracy']:>10.4f} {ev['mean_reward']:>10.2f} {ev['mean_gini']:>8.4f} {ev['mean_urgent_rate']:>10.4f} {ev['inference_ms']:>10.4f}")

report.append("\n\n=== SECTION 5: STATISTICAL SIGNIFICANCE ===")
report.append(f"DQN vs RF (accuracy): t={ttest_dqn_rf.statistic:.4f}, p={ttest_dqn_rf.pvalue:.6f}")
report.append(f"DQN vs REINFORCE (reward): t={ttest_dqn_reinforce.statistic:.4f}, p={ttest_dqn_reinforce.pvalue:.6f}")
report.append(f"DQN vs A2C (reward): t={ttest_dqn_a2c.statistic:.4f}, p={ttest_dqn_a2c.pvalue:.6f}")

report.append("\n\n=== SECTION 6: INTERPRETABILITY ANALYSIS ===")
for i, case in enumerate(interpretability_cases[:5]):
    report.append(f"\n--- Case {i+1} ---")
    report.append(f"Text: {case['complaint_text']}...")
    report.append(f"Topic: {case['topic']}")
    report.append(f"Correct dept: {case['correct_dept']}")
    report.append(f"Predicted: {case['predicted']} ({'CORRECT' if case['is_correct'] else 'WRONG'})")
    report.append(f"Urgency: {case['urgency']:.3f}")
    report.append(f"Risk flags: {case['risk_flags']}")
    report.append(f"Feature group importance:")
    for group, imp in case['group_importance'].items():
        report.append(f"  {group}: {imp:.4f}")
    report.append(f"Top 5 features:")
    for feat, weight in case['top_features'][:5]:
        report.append(f"  {feat}: {weight:.4f}")
    report.append(f"Q-values: {json.dumps({k: round(v, 2) for k, v in case['q_values'].items()})}")

report.append("\n\n=== SECTION 7: HYPERPARAMETER SENSITIVITY ===")
report.append(f"\n{'Config':<15} {'Train Reward':>15} {'Train Acc':>12} {'Eval Acc':>12}")
report.append("-" * 55)
for config, res in hp_results.items():
    report.append(f"{config:<15} {res['final_reward']:>15.2f} {res['final_accuracy']:>12.4f} {res['eval_accuracy']:>12.4f}")

report.append("\n\n=== SECTION 8: REWARD ABLATION STUDY ===")
report.append(f"\n{'Mode':<20} {'Accuracy':>10} {'Reward':>10} {'Gini':>8} {'Urgent%':>10}")
report.append("-" * 60)
for mode, res in ablation_results.items():
    report.append(f"{mode:<20} {res['accuracy']:>10.4f} {res['reward']:>10.2f} {res['gini']:>8.4f} {res['urgent_rate']:>10.4f}")

report.append("\n\n=== SECTION 9: TRAINING TIME COMPARISON ===")
report.append(f"Random Forest: {rf_train_time:.2f}s")
report.append(f"SVM: {svm_train_time:.2f}s")
report.append(f"DQN (300 eps): {dqn_train_time:.1f}s")
report.append(f"REINFORCE (300 eps): {reinforce_train_time:.1f}s")
report.append(f"A2C (300 eps): {a2c_train_time:.1f}s")

report.append("\n\n=== END OF REPORT ===")
report.append("Copy this entire text and send it back to Claude for paper generation.")

# Save report
report_text = "\n".join(report)
with open("results_report.txt", "w") as f:
    f.write(report_text)

print("\n" + "=" * 70)
print("DONE! Results saved to:")
print("  - results_report.txt  (copy this and send to Claude)")
print(f"  - {FIGURES_DIR}/*.png   (10 figures for the paper)")
print("=" * 70)
print("\nFirst 50 lines of report:")
print("\n".join(report[:50]))